## Into to pyspark

### Reading the data as Spark Dataframe 
### 



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
import os
import subprocess

# Jupyter kernels launched from an IDE/GUI don't source ~/.zshrc, so JAVA_HOME
# may be missing even if it's set in your shell profile. Resolve it via brew
# directly so this works regardless of how the kernel was started.
if "JAVA_HOME" not in os.environ:
    os.environ["JAVA_HOME"] = subprocess.check_output(["brew", "--prefix", "openjdk@17"], text=True).strip()
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("pyspark-demo").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark

In [ ]:
%%bash
uv run get_data.py

In [ ]:
# replace file path with your own if you have the dataset in a different location
df = spark.read.csv("data/y_amazon-google-large.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

In [ ]:
df.describe().show()

In [ ]:
from pyspark.sql import functions as f

spark.sparkContext.setJobDescription("min/max ds aggregation")
df.agg(f.min("ds").alias("min_ds"), f.max("ds").alias("max_ds")).show()

Spark UI: localcost:4040 (or 4041, 4042...) 

The SQL/DataFrame execution page renders one query as a DAG of physical operators, each annotated with live runtime metrics — this is Spark's most useful debugging view because it shows actual data volumes and timings, not just the plan.

Reading your DAG bottom-to-top (source → result):

┌────────────────────────┬──────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────┐
│          Node          │       Operator       │                                           What happened                                           │
├────────────────────────┼──────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Scan csv               │ reads the file       │ 3,055,000 ro MiB                                                        │
├────────────────────────┼──────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤
│ HashAggregate          │ per-partition        │ 11 tasks (onute a local min/max → 11 output rows, 3.6s total across     │
│ (partial)              │ min/max              │ tasks                                                                                             │
├────────────────────────┼──────────────────────┼─────────────────────────────────────────────────────────────────────────┤
│ Exchange               │ shuffle              │ those 11 partial rows get shuffled into a single partition so they can be combined — only 737     │
│                        │                      │ bytes moved,                                                            │
├────────────────────────┼──────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤
│ HashAggregate (final)  │ combines partials    │ merges the 1true global min/max row                                     │
├────────────────────────┼──────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤
│ AdaptiveSparkPlan      │ root                 │ wraps the whn took 412ms                                                │
└────────────────────────┴──────────────────────┴───────────────────────────────────────────────────────────────────────────────────────────────────┘

Why two HashAggregates

This is Spark's standard partial → shuffle → final aggregation pattern: instead of shuffling all 3M rows to one place to compute min/max, each partition
reduces its own chunk first (cheap, parallel), and only the tis) get shuffled and merged. That's why the Exchange step movedbytes, not megabytes.

The "Initial Plan" vs "Final Plan" you'll see in the text plan

Spark 4.x uses Adaptive Query Execution (AQE) by default — it plans optimistically, then can re-optimize using real stats once the shuffle (Exchange)
actually runs. That's why the raw plan text (visible if you ex an == Initial Plan == and == Final Plan ==; here they ended upthe same shape since there wasn't much to adapt.

The two job IDs (9, 10) at the bottom of the page

AQE submits each query stage (the plan segments separated by an Exchange) as its own Spark job when it materializes, rather than one job for the whole
query — that's why this single SQL execution shows up as two etead of one.

In [ ]:
%%bash
uv run get_data.py --url  'https://github.com/databricks/LearningSparkV2/blob/master/databricks-datasets/learning-spark-v2/mnm_dataset.csv'

In [ ]:
mnm = spark.read.csv("data/mnm_dataset.csv", header=True, inferSchema=True)
# mnm.count() # 99999

color_agg_mnm = mnm.groupBy("Color").count().orderBy("count", ascending=False)
color_agg_mnm.show()

In [ ]:
color_agg_mnm_pdf = color_agg_mnm.toPandas()
color_agg_mnm_pdf.plot.bar(x="Color", y="count", legend=False)

In [ ]:
sns.set_theme(style="whitegrid")
sns.barplot(x="Color", y="count", data=color_agg_mnm_pdf)
plt.title("M&M Color Distribution")


## RDD - Low level API

In [ ]:
# Extract the SparkContext from the session
sc = spark.sparkContext

# In Python
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30), ("TD", 35), ("Brooke", 25)])
# Use map and reduceByKey transformations with their lambda
# expressions to aggregate and then compute average
agesRDD = (
    dataRDD.map(lambda x: (x[0], (x[1], 1)))
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
    .map(lambda x: (x[0], x[1][0] / x[1][1]))
)

In [ ]:
agesRDD.take(5)

## Dataframe API

In [ ]:
from pyspark.sql.functions import avg

# Create a DataFrame
data_df = spark.createDataFrame(
    [("Brooke", 20), ("Denny", 31), ("Jules", 30), ("TD", 35), ("Brooke", 25)], ["name", "age"]
)
# Group the same names together, aggregate their ages, and compute an average
avg_df = data_df.groupBy("name").agg(avg("age"))
# Show the results of the final execution
avg_df.show()

In [ ]:
parquet_table = "avg_age_by_name"
avg_df.write.format("parquet").saveAsTable(parquet_table)

In [ ]:
from pyspark.sql import functions as f

dx = spark.read.format("parquet").table(parquet_table).withColumn("rnd", f.rand()).orderBy("rnd")
parquet_table2 = "avg_age_by_name2"
dx.write.format("parquet").saveAsTable(parquet_table2)

In [ ]:
spark.sql("CREATE DATABASE learn_spark_db")
spark.sql("USE learn_spark_db")

In [ ]:
spark.sql(
    "CREATE TABLE managed_us_delay_flights_tbl (date STRING, delay INT,distance INT, origin STRING, destination STRING) USING PARQUET"
)


In [ ]:
%%bash
uv run get_data.py --url 'https://github.com/databricks/LearningSparkV2/blob/master/databricks-datasets/learning-spark-v2/flights/departuredelays.csv'

In [ ]:
# Schema as defined in the preceding example
csv_file = "data/departuredelays.csv"
schema = "date STRING, delay INT, distance INT, origin STRING, destination STRING"
flights_df = spark.read.csv(csv_file, schema=schema)
flights_df.write.saveAsTable("managed_us_delay_flights_tbl", format="parquet", mode="append")

In [ ]:
df = spark.read.format("parquet").table("managed_us_delay_flights_tbl")
df.count()